# Ingest circuits.csv files
1. Read the files using spark dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

## Step 1 - Read the files using spark dataframe reader API


In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
landing_folder_path

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',    StringType(), True),
    StructField('url',          StringType(), True),
    StructField('circuitName',  StringType(), True),
    StructField('lat',          DoubleType(), True),
    StructField('long',         DoubleType(), True),
    StructField('locality',     StringType(), True),
    StructField('country',      StringType(), True)
])


In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'true')
        .option('mode', 'failFast')
#         .option('inferSchema', 'false')
        .schema(circuits_schema)
        .load(source_file)
)        


In [0]:
display(circuits_df)

### Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)           

In [0]:
display(circuits_final_df)

## Write to bronze delta table


In [0]:
( 
    circuits_final_df
            .write
            .format('delta')
            .mode('overwrite')
            .saveAsTable(table_name)
)

In [0]:
display(spark.read.table(table_name))